# Strands Agents with AgentCore Memory (Long term memory via Tool)

## Introduction

This notebook demonstrates how to implement a **multi-agent system with shared long-term memory** using AgentCore Memory and the Strands framework. We will explore how multiple specialized agents can work together while accessing a common long-term memory store with dedicated namespaces for each agent.

### Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Long term Conversational                                                         |
| Agent type          | Travel Booking Assistant                                                         |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Sonnet 3.7                                                      |
| Tutorial components | AgentCore User Preferences Memory Extraction, Tool for storing and retrieving Memory              |
| Example complexity  | Intermediate                                                                     |

You will learn:

- How to set up a shared memory resource with a long-term memory strategy
- Creating specialized agents with access to their own memory namespaces
- Implementing a coordinator agent that delegates to specialized agents
- Leveraging structured memory namespaces for agent specialization

## Scenario context

In this example, we'll create a **Travel Planning System** with:
1. A Flight Booking Assistant with long-term memory of travel preferences and history
2. A Hotel Booking Assistant with long-term memory of accommodation preferences
3. A Travel Coordinator that orchestrates these specialized agents

Each specialized agent will access its own namespace within a common memory store, enabling them to build persistent understanding of user preferences over time. This approach demonstrates how complex domains can be broken down into specialized agents that share a memory infrastructure but maintain their own areas of expertise.

## Architecture
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## Prerequisites

To execute this tutorial you will need:
- Python 3.10+
- AWS credentials with Amazon Bedrock AgentCore Memory permissions
- Amazon Bedrock AgentCore SDK

Let's get started by setting up our environment and creating our shared long-term memory resource!

## Step 1: Environment set up
Let's begin importing all the necessary libraries and defining the clients to make this notebook work.

In [1]:
!pip install -qr requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import logging
import time
from datetime import datetime
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("travel-assistant")

Define the region with the appropiate permissions for Amazon Bedrock models and AgentCore

In [ ]:
region = "us-west-2" # Replace with your AWS region
MODEL_ID= "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

## Step 2: Creating Shared Memory Resource
In this section, we'll create common long-term memory store with dedicated namespaces for each agent.

In [9]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

client = MemoryClient(region_name=region)
memory_name = "TravelAgent_LTM"
memory_id = None

2025-08-07 10:57:49 - INFO - Initialized MemoryClient for control plane: us-east-1, data plane: us-east-1


In [10]:
from botocore.exceptions import ClientError

try:
    print("Creating Memory with Long-Term Strategy...")
    # Create the memory resource with a single long-term memory strategy
    # The {actorId} placeholder will be dynamically replaced with the actual actor ID
    memory = client.create_memory_and_wait(
        name=memory_name,
        description="Travel Agent with Long-Term Memory",
        strategies=[{
            StrategyType.USER_PREFERENCE.value: {
                "name": "UserPreferences",
                "description": "Captures user preferences",
                "namespaces": ["travel/{actorId}/preferences"]
            }
        }],
        event_expiry_days=7,  # Short-term conversation expires after 7 days
        max_wait=300,
        poll_interval=10
    )

    # Extract and print the memory ID
    memory_id = memory['id']
    print(f"Memory created successfully with ID: {memory_id}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # If memory already exists, retrieve its ID
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Handle any errors during memory creation
    logger.info(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()
    # Cleanup on error - delete the memory if it was partially created
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"Failed to clean up memory: {cleanup_error}")

2025-08-07 10:57:51 - INFO - Loading cached SSO token for agentcore


Creating Memory with Long-Term Strategy...


2025-08-07 10:57:52 - INFO - Created memory: TravelAgent_LTM-tiRFyp9Rgg
2025-08-07 10:57:52 - INFO - Created memory TravelAgent_LTM-tiRFyp9Rgg, waiting for ACTIVE status...
2025-08-07 11:00:39 - INFO - Memory TravelAgent_LTM-tiRFyp9Rgg is now ACTIVE (took 166 seconds)


Memory created successfully with ID: TravelAgent_LTM-tiRFyp9Rgg


### Understanding Long-Term Memory Strategy

The memory resource we're creating uses AgentCore Memory's long-term memory capabilities with a single user preference strategy:

1. **User Preference Memory Strategy**: Automatically extracts and consolidates user preferences mentioned in conversations
2. **Actor-based Namespaces**: Uses the actor ID in the namespace path to create separate spaces for each agent
3. **Memory Persistence**: Unlike short-term memory that expires, extracted preferences persist beyond conversation expiry

With the namespace pattern `travel/{actorId}/preferences`, each specialized agent will have its own unique namespace based on its actor ID:
- Flight agent will access: `travel/flight-user-TIMESTAMP/preferences`
- Hotel agent will access: `travel/hotel-user-TIMESTAMP/preferences`

This allows each agent to maintain its own specialized knowledge while using a common memory infrastructure.

### Setting Up Agent Identities

In [11]:
# Create unique actor IDs for each specialized agent but share the session ID
flight_actor_id = f"flight-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
hotel_actor_id = f"hotel-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"travel-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"
flight_namespace = f"travel/{flight_actor_id}/preferences"
hotel_namespace = f"travel/{hotel_actor_id}/preferences"

In [12]:
# Import the necessary components
from strands import Agent, tool
from strands_tools.agent_core_memory import AgentCoreMemoryToolProvider

### Step 3: Create Memory Hook Provider

This step defines our custom `MemoryHookProvider` class that automates memory operations. Hooks are special functions that run at specific points in an agent's execution lifecycle. The memory hook we're creating serves one primary function:

1. **Save Memories**: Stores new conversations after the agent responds

This creates a seamless memory experience without manual management.

In [13]:
class MemoryHookProvider(HookProvider):
    """Hook provider for automatic memory management"""
    
    def __init__(self, memory_id: str, client: MemoryClient, actor_id: str, session_id: str):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.session_id = session_id
    
    def save_memories(self, event: AfterInvocationEvent):
        """Save conversation after agent response"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2:
                # Get last user and assistant messages
                user_msg = None
                assistant_msg = None
                
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not assistant_msg:
                        assistant_msg = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not user_msg and "toolResult" not in msg["content"][0]:
                        user_msg = msg["content"][0]["text"]
                        break
                
                if user_msg and assistant_msg:
                    # Save conversation
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=self.actor_id,
                        session_id=self.session_id,
                        messages=[(user_msg, "USER"), (assistant_msg, "ASSISTANT")]
                    )
                    logger.info("Saved conversation to memory")
                    
        except Exception as e:
            logger.error(f"Failed to save memories: {e}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        """Register memory hooks"""
        registry.add_callback(AfterInvocationEvent, self.save_memories)
        logger.info("Memory hooks registered")

### Creating Specialized Agents with Memory Access

Next, we'll define system prompts for our specialized agents.

In [14]:
# System prompt for the hotel booking specialist
HOTEL_BOOKING_PROMPT = f"""You are a hotel booking assistant. Help customers find hotels, make reservations, and answer questions about accommodations and amenities. 
Provide clear information about availability, pricing, and booking procedures in a friendly, helpful manner.Ask max two questions per turn. Keep the messages short, don't overwhelm the customer."""

# System prompt for the flight booking specialist
FLIGHT_BOOKING_PROMPT = f"""You are a flight booking assistant. Help customers find flights, make reservations, and answer questions about airlines, routes, and travel policies. 
Provide clear information about flight availability, pricing, schedules, and booking procedures in a friendly, helpful manner.Ask max two questions per turn. Keep the messages short, don't overwhelm the customer."""

### Implementing Agent Tools
Now we'll implement our specialized agents as tools that can be used by the coordinator agent:

In [16]:
@tool
def flight_booking_assistant(query: str) -> str:
    """
    Process and respond to flight booking queries.

    Args:
        query: A flight-related question about bookings, schedules, airlines, or travel policies

    Returns:
        Detailed flight information, booking options, or travel advice
    """
    try:
        provider_flight = AgentCoreMemoryToolProvider(
            memory_id=memory_id,      # Required
            actor_id=flight_actor_id, # Required
            session_id=session_id,    # Required
            region = region,
            namespace=flight_namespace
        )
        
        flight_memory_hooks = MemoryHookProvider(
            memory_id=memory_id,
            client=client,
            actor_id=flight_actor_id,
            session_id=session_id
        )

        flight_agent = Agent(tools=provider_flight.tools,
                             hooks=[flight_memory_hooks],
                             model=MODEL_ID,
                             system_prompt=FLIGHT_BOOKING_PROMPT
                             )

        # Call the agent and return its response
        response = flight_agent(query)
        return str(response)
    except Exception as e:
        return f"Error in flight booking assistant: {str(e)}"

@tool
def hotel_booking_assistant(query: str) -> str:
    """
    Process and respond to hotel booking queries.

    Args:
        query: A hotel-related question about accommodations, amenities, or reservations

    Returns:
        Detailed hotel information, booking options, or accommodation advice
    """
    try:
        provider_hotel = AgentCoreMemoryToolProvider(
            memory_id=memory_id,      
            actor_id=hotel_actor_id, 
            session_id=session_id,   
            region = region,
            namespace=hotel_namespace
        )

        hotel_memory_hooks = MemoryHookProvider(
            memory_id=memory_id,
            client=client,
            actor_id=hotel_actor_id,
            session_id=session_id
        )

        hotel_booking_agent = Agent(tools=provider_hotel.tools, hooks=[hotel_memory_hooks],model=MODEL_ID, system_prompt=HOTEL_BOOKING_PROMPT)
        
        # Call the agent and return its response
        response = hotel_booking_agent(query)
        return str(response)
    except Exception as e:
        return f"Error in hotel booking assistant: {str(e)}"

### Creating the Coordinator Agent

Finally, we'll create the main travel planning agent that coordinates between these specialized tools:

In [17]:
# System prompt for the coordinator agent
TRAVEL_AGENT_SYSTEM_PROMPT = """
You are a comprehensive travel planning assistant that coordinates between specialized tools:
- For flight-related queries (bookings, schedules, airlines, routes) → Use the flight_booking_assistant tool
- For hotel-related queries (accommodations, amenities, reservations) → Use the hotel_booking_assistant tool
- For complete travel packages → Use both tools as needed to provide comprehensive information
- For general travel advice or simple travel questions → Answer directly

Each agent will have its own memory in case the user asks about historic data.
When handling complex travel requests, coordinate information from both tools to create a cohesive travel plan.
Provide clear organization when presenting information from multiple sources. \
Ask max two questions per turn. Keep the messages short, don't overwhelm the customer.
"""

In [18]:
travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant]
)

## Testing the Multi-Agent Memory System

Let's test our multi-agent system with a travel planning scenario:

### Optional: Hydrate the long term memory of the flight booking assistant
Uncomment the following cell if you want to hydrate the flight booking agent.

In [20]:
flight_previous_messages = [
    ("Hi, I'm Sarah", "USER"),
    ("Hello Sarah! Welcome to FlightAssist. How can I help you with your travel plans today?", "ASSISTANT"),
    ("I'm looking to book a flight from New York to London sometime next month.", "USER"),
    ("I'd be happy to help you find flights from New York to London for next month. Do you have specific dates in mind, or are you flexible?", "ASSISTANT"),
    ("I'm thinking around the 15th to the 25th, but I can be a bit flexible.", "USER"),
    ("Great! That gives us some room to find the best options. Do you have any preferences regarding airlines or flight times?", "ASSISTANT"),
    ("I definitely prefer direct flights if possible. I really don't like layovers.", "USER"),
    ("I completely understand your preference for direct flights. Layovers can be inconvenient. Fortunately, there are several airlines offering direct flights between New York and London, including British Airways, American Airlines, Delta, and Virgin Atlantic.", "ASSISTANT"),
    ("That's good to hear. I've had good experiences with British Airways in the past.", "USER"),
    ("British Airways does offer excellent service on transatlantic routes. I'll keep that in mind when searching for options. Do you have any seating preferences or other requirements for your flight?", "ASSISTANT"),
    ("I always try to get an aisle seat. I like being able to get up without disturbing others, especially on long flights.", "USER"),
    ("An aisle seat is a great choice for long-haul flights like New York to London. I'll note your preference for aisle seating. Would you prefer to fly in the morning, afternoon, or evening?", "ASSISTANT"),
    ("I prefer overnight flights for long journeys. It helps me adjust to the time difference better.", "USER"),
    ("Overnight flights are indeed a smart choice for eastbound transatlantic travel. They allow you to arrive in London in the morning and help minimize jet lag. British Airways, Delta, and American all offer evening departures from New York that arrive in London the next morning.", "ASSISTANT"),
    ("Perfect! And I'm also wondering about baggage allowances since I'll be staying for about a week.", "USER"),
    ("For a week-long trip, most travelers find that a standard checked bag plus a carry-on is sufficient. British Airways typically allows one free checked bag on transatlantic flights in economy class, plus a carry-on and personal item. Would you like me to check the specific allowances for your preferred dates?", "ASSISTANT")
]

print("\nHydrating memories with previous conversations...")

# Save the conversation history to short-term memory
initial = client.create_event(
    memory_id=memory_id,
    actor_id=flight_actor_id,
    session_id=session_id,
    messages=flight_previous_messages,
)
print("✓ Conversation saved in short term memory")


Hydrating memories with previous conversations...


2025-08-07 11:08:44 - INFO - Created event: 0000001754536123000#806ce53e


✓ Conversation saved in short term memory


In [21]:
travel_agent("Hello, I would like to book a trip from LA to Madrid. From July 1 to August 2.")

2025-08-07 11:08:46 - INFO - Creating Strands MetricsClient
2025-08-07 11:08:46 - INFO - Loading cached SSO token for agentcore


Hello! I'd be happy to help you plan your trip from LA to Madrid from July 1 to August 2. I'll need to gather information about both your flight and hotel preferences.

Let's start with your flight:
Tool #1: flight_booking_assistant


2025-08-07 11:08:52 - INFO - Memory hooks registered
2025-08-07 11:08:52 - INFO - Loading cached SSO token for agentcore


I'd be happy to help you find flight options from Los Angeles to Madrid for your trip. Let me check if I have any saved information about flights between these locations or your preferences from previous conversations.
Tool #1: agent_core_memory
I see you're looking for flights from LA to Madrid from July 1 to August 2. I found a previous preference indicating you prefer direct flights, but I'll need more details to provide accurate options.

Could you please confirm:
1. Are you still looking for direct flights only, or are you open to flights with layovers?
2. Do you have any airline preferences or are you looking for the most economical options?

2025-08-07 11:09:01 - INFO - Created event: 0000001754536140000#8febac37
2025-08-07 11:09:01 - INFO - Saved conversation to memory


...

2025-08-07 11:09:03 - INFO - Memory hooks registered
2025-08-07 11:09:03 - INFO - Loading cached SSO token for agentcore


I'd be happy to help you find hotel options in Madrid from July 1 to August 2. Let me see if I have any information about hotels in Madrid from previous interactions.
Tool #1: agent_core_memory
I don't have any stored information about hotels in Madrid yet. To help you find the best options, I'll need to know a bit more:

1. What's your budget range per night for the hotel?
2. Do you have any specific preferences for the hotel type or location within Madrid (like city center, near attractions, etc.)?

Once I have this information, I can provide you with appropriate hotel recommendations for your month-long stay in Madrid from July 1 to August 2.

2025-08-07 11:09:12 - INFO - Created event: 0000001754536152000#abb42a2c
2025-08-07 11:09:12 - INFO - Saved conversation to memory


To provide you with the best options for your trip from LA to Madrid (July 1 - August 2), I need a few more details:

For your flights:
1. Do you prefer direct flights only or are you open to flights with layovers?

For your hotel stay:
2. What's your budget range per night for the hotel in Madrid?

Once you provide these details, I can offer you specific recommendations for both your flight and accommodation needs.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "To provide you with the best options for your trip from LA to Madrid (July 1 - August 2), I need a few more details:\n\nFor your flights:\n1. Do you prefer direct flights only or are you open to flights with layovers?\n\nFor your hotel stay:\n2. What's your budget range per night for the hotel in Madrid?\n\nOnce you provide these details, I can offer you specific recommendations for both your flight and accommodation needs."}]}, metrics=EventLoopMetrics(cycle_count=3, tool_metrics={'flight_booking_assistant': ToolMetrics(tool={'toolUseId': 'tooluse_N_TzS7_JTmqN8UD8eP4sdg', 'name': 'flight_booking_assistant', 'input': {'query': 'I need flight options from LA to Madrid from July 1 to August 2'}}, call_count=1, success_count=1, error_count=0, total_time=8.676472902297974), 'hotel_booking_assistant': ToolMetrics(tool={'toolUseId': 'tooluse_vC50Oc_USPiwuO9xmP5asQ', 'name': 'hotel_booking_assistant', 'inp

In [22]:
travel_agent("I prefer direct flights with Iberia")

Thank you for letting me know your preference for direct flights with Iberia. I'll search for those options right away.
Tool #3: flight_booking_assistant


2025-08-07 11:10:44 - INFO - Memory hooks registered
2025-08-07 11:10:44 - INFO - Loading cached SSO token for agentcore


I'd be happy to help you find direct flights with Iberia from Los Angeles to Madrid. Let me check if I have any information about similar flight searches in my memory first.
Tool #1: agent_core_memory
I'll help you find direct flights with Iberia from LA to Madrid for your dates. Let me save this request to remember your preferences.
Tool #2: agent_core_memory
To find the specific flights you're looking for, I'll need to confirm a few details:

1. Are you looking for one-way or round-trip flights?
2. How many passengers will be traveling?

Once I have this information, I can provide you with available Iberia direct flights from Los Angeles (LAX) to Madrid (MAD) departing on July 1st and returning on August 2nd.

2025-08-07 11:10:57 - INFO - Created event: 0000001754536257000#e2592e4a
2025-08-07 11:10:57 - INFO - Saved conversation to memory


I need a few more details to find the perfect Iberia direct flights for your trip:

1. Are you looking for a one-way or round-trip flight?
2. How many passengers will be traveling?

Also, do you have any preferences for your hotel in Madrid? A budget range or preferred area would help me find suitable accommodations for your month-long stay.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'I need a few more details to find the perfect Iberia direct flights for your trip:\n\n1. Are you looking for a one-way or round-trip flight?\n2. How many passengers will be traveling?\n\nAlso, do you have any preferences for your hotel in Madrid? A budget range or preferred area would help me find suitable accommodations for your month-long stay.'}]}, metrics=EventLoopMetrics(cycle_count=5, tool_metrics={'flight_booking_assistant': ToolMetrics(tool={'toolUseId': 'tooluse_GSWUuSiHTta_q_8j-wEbvw', 'name': 'flight_booking_assistant', 'input': {'query': 'I need direct flights with Iberia from LA to Madrid from July 1 to August 2'}}, call_count=2, success_count=2, error_count=0, total_time=22.55616331100464), 'hotel_booking_assistant': ToolMetrics(tool={'toolUseId': 'tooluse_vC50Oc_USPiwuO9xmP5asQ', 'name': 'hotel_booking_assistant', 'input': {'query': 'I need hotel options in Madrid from July 1 to Augus

In [23]:
travel_agent("I would like a flight in the morning, in economy")

Thank you for the additional information. I'll look for morning economy flights with Iberia from LA to Madrid.
Tool #4: flight_booking_assistant


2025-08-07 11:11:07 - INFO - Memory hooks registered
2025-08-07 11:11:07 - INFO - Loading cached SSO token for agentcore


I'd be happy to help you find direct flights with Iberia from LA to Madrid. Let me check if I have any saved information about this route first.
Tool #1: agent_core_memory
I'll help you find direct flights with Iberia from LA to Madrid with the specifics you requested. Let me save these preferences first.
Tool #2: agent_core_memory
I need to check our flight database for your request, but unfortunately I don't have direct access to real-time flight inventory. To proceed with finding your Iberia direct flights:

1. Date: July 1 (outbound, morning departure) to August 2 (return)
2. Route: Los Angeles (LAX) to Madrid (MAD)
3. Airline: Iberia
4. Class: Economy
5. Preference: Direct flights only

Would you like me to:
1. Check if there are any baggage requirements or restrictions you should know about for this route?
2. Provide information about Iberia's services on transatlantic flights?

2025-08-07 11:11:22 - INFO - Created event: 0000001754536282000#d963cc33
2025-08-07 11:11:22 - INFO - Saved conversation to memory


I've checked for your morning economy direct flights with Iberia from LA to Madrid. While I don't have access to real-time flight inventory, I can confirm I've noted your preferences:

- Direct flights with Iberia
- Economy class
- Morning departure on July 1
- Return on August 2
- Route: Los Angeles (LAX) to Madrid (MAD)

Would you like information about:
1. Iberia's baggage requirements for this route?
2. Services offered on their transatlantic flights?

Also, to help with your hotel booking in Madrid, could you share your budget range or any preferences for location within the city?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "I've checked for your morning economy direct flights with Iberia from LA to Madrid. While I don't have access to real-time flight inventory, I can confirm I've noted your preferences:\n\n- Direct flights with Iberia\n- Economy class\n- Morning departure on July 1\n- Return on August 2\n- Route: Los Angeles (LAX) to Madrid (MAD)\n\nWould you like information about:\n1. Iberia's baggage requirements for this route?\n2. Services offered on their transatlantic flights?\n\nAlso, to help with your hotel booking in Madrid, could you share your budget range or any preferences for location within the city?"}]}, metrics=EventLoopMetrics(cycle_count=7, tool_metrics={'flight_booking_assistant': ToolMetrics(tool={'toolUseId': 'tooluse_Hw1VTo_6SWafeauxjMAIAQ', 'name': 'flight_booking_assistant', 'input': {'query': 'I need direct flights with Iberia from LA to Madrid on July 1 (morning departure) and returning on 

In [24]:
travel_agent("I would like to fly from SNA, and return 15 days later")

Thank you for the updated information. Let me search for flights from SNA (John Wayne Airport in Orange County) with the revised return date.
Tool #5: flight_booking_assistant


2025-08-07 11:20:12 - INFO - Memory hooks registered
2025-08-07 11:20:12 - INFO - Loading cached SSO token for agentcore


I'd be happy to help you find direct flights from Orange County (SNA) to Madrid with Iberia. Let me check if I have any information about these flights in my memory.
Tool #1: agent_core_memory
I need to check if there's any specific information about direct flights from SNA to Madrid with Iberia. Let me search for more specific information.
Tool #2: agent_core_memory
I need to inform you about an important detail regarding your flight request. Based on my search, I don't have specific information about direct Iberia flights from SNA (Orange County) to Madrid. 

Important to note:
- Iberia typically doesn't operate direct flights from Orange County (SNA) to Madrid
- Most international flights to Madrid from Southern California depart from LAX

Would you be willing to:
1. Consider flights with a connection instead of direct?
2. Or consider departing from LAX instead of SNA?

2025-08-07 11:20:30 - INFO - Created event: 0000001754536830000#12c7a995
2025-08-07 11:20:30 - INFO - Saved conversation to memory


I need to inform you about an important detail regarding your flight request. Based on my search, Iberia typically doesn't operate direct flights from Orange County (SNA) to Madrid. Most international flights to Madrid from Southern California depart from LAX.

Would you prefer:
1. Flights with connections from SNA to Madrid? 
2. Direct flights from LAX to Madrid instead?

Once we clarify your departure airport preference, I can also help with your hotel booking for your 15-day stay in Madrid (July 1-16).

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "I need to inform you about an important detail regarding your flight request. Based on my search, Iberia typically doesn't operate direct flights from Orange County (SNA) to Madrid. Most international flights to Madrid from Southern California depart from LAX.\n\nWould you prefer:\n1. Flights with connections from SNA to Madrid? \n2. Direct flights from LAX to Madrid instead?\n\nOnce we clarify your departure airport preference, I can also help with your hotel booking for your 15-day stay in Madrid (July 1-16)."}]}, metrics=EventLoopMetrics(cycle_count=9, tool_metrics={'flight_booking_assistant': ToolMetrics(tool={'toolUseId': 'tooluse_OJNInQ30Tdqs45zhg-TGHg', 'name': 'flight_booking_assistant', 'input': {'query': 'I need direct flights with Iberia from SNA (Orange County) to Madrid on July 1 (morning departure) and returning 15 days later (July 16) in economy class'}}, call_count=4, success_count=4

## Testing Memory Persistence

To test if our memory system is working correctly, we'll create a new instance of the travel agent and see if it can access the previously stored information:

In [25]:
time.sleep(60) # Let's give the memory some time to process the events..
# Create a new instance of the travel agent
new_travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant]
)

# Ask about previous conversations
new_travel_agent("Can you remind me about flight preferences?")

2025-08-07 11:24:43 - INFO - Loading cached SSO token for agentcore


I'd be happy to remind you about your flight preferences. To provide accurate information, I need to access your saved flight preferences from our system.
Tool #1: flight_booking_assistant


2025-08-07 11:24:48 - INFO - Memory hooks registered
2025-08-07 11:24:48 - INFO - Loading cached SSO token for agentcore


I'd be happy to check if there are any saved flight preferences in your records. Let me look that up for you.
Tool #1: agent_core_memory
Based on your saved flight preferences, I can see you have the following preferences:

1. Seating: You prefer aisle seats, especially on long-haul flights
2. Class: Economy class travel
3. Timing: You prefer overnight flights for long international travel and morning departures
4. Airlines: You've had good experiences with British Airways for international flights and have shown interest in Iberia Airlines
5. Routes: You prefer direct flights without layovers
6. Travel Plans: You've previously searched for flights between Los Angeles/SNA and Madrid during summer (July 1-16)

Is there anything specific about these preferences you'd like to know more about or would you like to update any of them?

2025-08-07 11:24:58 - INFO - Created event: 0000001754537098000#844c33b6
2025-08-07 11:24:58 - INFO - Saved conversation to memory


Here are your saved flight preferences:

1. Seating: Aisle seats, especially for long-haul flights
2. Class: Economy class travel
3. Timing: Overnight flights for long international trips, morning departures generally
4. Airlines: British Airways for international flights, with some interest in Iberia Airlines
5. Routes: Direct flights preferred (no layovers)
6. Previous Search: Los Angeles/SNA to Madrid (July 1-16)

Is there anything specific about these preferences you'd like to adjust or would you like more details about any of them?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Here are your saved flight preferences:\n\n1. Seating: Aisle seats, especially for long-haul flights\n2. Class: Economy class travel\n3. Timing: Overnight flights for long international trips, morning departures generally\n4. Airlines: British Airways for international flights, with some interest in Iberia Airlines\n5. Routes: Direct flights preferred (no layovers)\n6. Previous Search: Los Angeles/SNA to Madrid (July 1-16)\n\nIs there anything specific about these preferences you'd like to adjust or would you like more details about any of them?"}]}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'flight_booking_assistant': ToolMetrics(tool={'toolUseId': 'tooluse_tn8GEBmCTv2flWQ0EKIV7g', 'name': 'flight_booking_assistant', 'input': {'query': "What are the user's saved flight preferences?"}}, call_count=1, success_count=1, error_count=0, total_time=10.870857954025269)}, cycle_durations=[3.4978

## Summary

In this notebook, we've demonstrated:

1. How to create a shared memory resource for multiple agents
2. How to implement specialized agents as tools with memory access
3. How to coordinate between multiple agents while maintaining conversation context
4. How memory persists across different agent instances

This multi-agent architecture with shared memory provides a powerful approach for building complex conversational AI systems that can handle specialized domains while maintaining a cohesive user experience.

## Clean up
Let's delete the memory to clean up the resources used in this notebook.

In [ ]:
# client.delete_memory_and_wait(
#        memory_id = memory_id,
#        max_wait = 300,
#        poll_interval =10
#)